## Deployment Summary

O modelo foi exportado com sucesso para o formato vLLM. 

### Para usar em produção:

```bash
# Iniciar servidor vLLM
python -m vllm.entrypoints.openai.api_server \
    --model ../results/vllm-model \
    --dtype half \
    --gpu-memory-utilization 0.9
```

### Alternativa com Docker:

```dockerfile
FROM nvcr.io/nvidia/cuda:12.1.0-runtime-ubuntu22.04

WORKDIR /app
COPY ../results/vllm-model /app/model
COPY requirements.txt .

RUN pip install -r requirements.txt

CMD ["python", "-m", "vllm.entrypoints.openai.api_server", \
     "--model", "/app/model", \
     "--dtype", "half", \
     "--gpu-memory-utilization", "0.9", \
     "--host", "0.0.0.0", \
     "--port", "8000"]
```

In [ ]:
# Validate the exported model
try:
    from vllm import LLM, SamplingParams
    
    print("Loading model with vLLM...")
    llm = LLM(
        model=export_dir,
        dtype="half",
        gpu_memory_utilization=0.8,
        tensor_parallel_size=1,
    )
    
    # Test inference
    sampling_params = SamplingParams(
        temperature=0.7,
        top_p=0.95,
        max_tokens=256,
    )
    
    test_prompt = "Context: Paris is the capital of France.\nQuestion: What is the capital of France?\nAnswer:"
    
    outputs = llm.generate([test_prompt], sampling_params)
    
    print("✓ vLLM inference test successful!")
    print(f"\nTest prompt:\n{test_prompt}")
    print(f"\nGenerated response:\n{outputs[0].outputs[0].text}")
    
except Exception as e:
    print(f"⚠ vLLM validation error (may be expected if CUDA is unavailable): {e}")

## Validate Export

In [ ]:
import json
import shutil

# Create vLLM export directory
export_dir = "../results/vllm-model"
if os.path.exists(export_dir):
    shutil.rmtree(export_dir)
os.makedirs(export_dir, exist_ok=True)

# Save model in vLLM format
model.save_pretrained(export_dir, safe_serialization=True)
tokenizer.save_pretrained(export_dir)

# Create vLLM config file
vllm_config = {
    "model_type": "gemma",
    "tensor_parallel_size": 1,
    "pipeline_parallel_size": 1,
    "gpu_memory_utilization": 0.9,
    "max_seq_len": 2048,
    "quantization": "awq",  # Can be awq, gptq, or disabled
}

config_path = os.path.join(export_dir, "vllm_config.json")
with open(config_path, "w") as f:
    json.dump(vllm_config, f, indent=2)

print(f"✓ Model exported to {export_dir}")
print(f"✓ vLLM config created")

## Convert to vLLM Format

vLLM requer um formato específico com config otimizado para serving. Vamos converter e salvar em um formato vLLM-compatível.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load the fine-tuned model
model_path = "../results/fine-tuned-model"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model from {model_path}...")
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_path)

print(f"✓ Model loaded on {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## Load Fine-tuned Model

In [ ]:
import subprocess
import sys
import os

# Install vLLM and dependencies
packages = [
    "vllm",
    "transformers",
    "peft",
    "torch",
    "accelerate",
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✓ vLLM and dependencies installed!")

# vLLM Export and Deployment

Este notebook exporta o modelo fine-tuned para o formato vLLM, otimizando-o para serving de alta performance em produção.

## Pipeline
1. Carregar modelo fine-tuned
2. Exportar para formato vLLM
3. Validar exportação
4. Preparar para deployment